# Dataset Builder — YOLOv8 (ArchGuard)

Pipeline para gerar o dataset de treino a partir dos ícones oficiais AWS, Azure e GCP.

**Classes STRIDE:**
| ID | Classe |
|----|--------|
| 0 | actor |
| 1 | gateway |
| 2 | compute |
| 3 | database |
| 4 | storage |
| 5 | message_bus |

### Datasets:
- [Kaggle — Software Architecture Dataset](https://www.kaggle.com/datasets/carlosrian/software-architecture-dataset)
- [AWS Architecture Icons](https://aws.amazon.com/pt/architecture/icons/)
- [Azure Architecture Icons](https://learn.microsoft.com/en-us/azure/architecture/icons/)
- [Google Cloud Icons](https://cloud.google.com/icons?hl=pt_br)

In [3]:
!pip install cairosvg -q


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [4]:
"""
=============================================================
CELL 1: Imports e Constantes
=============================================================
"""
import os
import random
import shutil
from pathlib import Path
from collections import defaultdict

# Fix para macOS: garante que dylibs do Homebrew são encontradas
os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

import cairosvg
from PIL import Image

# ───────────────────────────────────────────────────────────
# PATHS
# ───────────────────────────────────────────────────────────
# O notebook roda em jupyter/, então subimos um nível para a raiz do projeto
PROJECT_ROOT = Path(os.getcwd()).parent
DATASETS_DIR = PROJECT_ROOT / "datasets"

AWS_ICONS = DATASETS_DIR / "aws-icons"
AZURE_ICONS = DATASETS_DIR / "azure-icons" / "Icons"
GCP_CATEGORY = DATASETS_DIR / "google-icons" / "Category"
GCP_UNIQUE = DATASETS_DIR / "google-icons" / "Unique"

OUTPUT_DIR = DATASETS_DIR / "archguard"

IMG_SIZE = 640
SPLIT_RATIO = 0.8  # 80% train, 20% val
RANDOM_SEED = 42

# ───────────────────────────────────────────────────────────
# CLASSES STRIDE
# ───────────────────────────────────────────────────────────
CLASS_NAMES = ['actor', 'gateway', 'compute', 'database', 'storage', 'message_bus']

# Mapeamento: nome normalizado da pasta → class_id
# (Aplica-se a AWS Architecture/Resource, Azure e GCP Category)
FOLDER_TO_CLASS = {
    # actor (0)
    "identity": 0,
    "security": 0,
    "general-icons": 0,   # AWS General → ícones de User/Client/Mobile
    "general": 0,         # Azure general
    # gateway (1)
    "networking-content-delivery": 1,
    "networking": 1,
    # compute (2)
    "compute": 2,
    "containers": 2,
    "serverless computing": 2,
    # database (3)
    "database": 3,
    "databases": 3,
    # storage (4)
    "storage": 4,
    # message_bus (5)
    "app-integration": 5,
    "integration": 5,
    "integration services": 5,
}

# ───────────────────────────────────────────────────────────
# AWS Security-Identity-Compliance: keyword-based (não mapeamento direto)
# ───────────────────────────────────────────────────────────
AWS_SECURITY_KEYWORDS_GATEWAY = ["waf", "shield", "kms", "firewall", "network-firewall"]
AWS_SECURITY_KEYWORDS_ACTOR = ["cognito", "directory", "iam", "identity"]

# ───────────────────────────────────────────────────────────
# Google Unique: keyword no nome da pasta → class_id
# ───────────────────────────────────────────────────────────
GCP_UNIQUE_MAPPING = {
    "Compute Engine": 2,
    "GKE": 2,
    "Distributed Cloud": 2,
    "Cloud SQL": 3,
    "Cloud Spanner": 3,
    "AlloyDB": 3,
    "BigQuery": 3,
    "Cloud Storage": 4,
    "Hyperdisk": 4,
    "Apigee": 1,
    "Security Command Center": 0,
    "Mandiant": 0,
    "Security Operations": 0,
    "Threat Intelligence": 0,
    "Pub/Sub": 5,
    "Cloud Tasks": 5,
}

print(f"✅ Constantes carregadas. PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   DATASETS_DIR: {DATASETS_DIR}")
print(f"   AWS exists: {AWS_ICONS.exists()}")
print(f"   Azure exists: {AZURE_ICONS.exists()}")
print(f"   GCP Category exists: {GCP_CATEGORY.exists()}")
print(f"   GCP Unique exists: {GCP_UNIQUE.exists()}")
print(f"   Classes: {CLASS_NAMES}")
print(f"   Output: {OUTPUT_DIR}")

✅ Constantes carregadas. PROJECT_ROOT: /Users/pedromarquardt/dev/estudos/tech-challenger/archguard-ai/ArchGuard-AI
   DATASETS_DIR: /Users/pedromarquardt/dev/estudos/tech-challenger/archguard-ai/ArchGuard-AI/datasets
   AWS exists: True
   Azure exists: True
   GCP Category exists: True
   GCP Unique exists: True
   Classes: ['actor', 'gateway', 'compute', 'database', 'storage', 'message_bus']
   Output: /Users/pedromarquardt/dev/estudos/tech-challenger/archguard-ai/ArchGuard-AI/datasets/archguard


In [5]:
"""
=============================================================
CELL 2: Funções Utilitárias
=============================================================
"""

def convert_svg_to_png(svg_path: Path, output_path: Path, size: int = IMG_SIZE) -> bool:
    """Converte SVG para PNG 640x640 com fundo BRANCO (crítico para YOLOv8)."""
    try:
        cairosvg.svg2png(
            url=str(svg_path),
            write_to=str(output_path),
            output_width=size,
            output_height=size,
            background_color="white"  # Evita fundo preto no OpenCV
        )
        return True
    except Exception as e:
        print(f"  ⚠️  SVG corrompido ou inválido: {svg_path.name} → {e}")
        return False


def resize_image_to_canvas(img_path: Path, output_path: Path, size: int = IMG_SIZE) -> bool:
    """Redimensiona PNG/JPG para canvas branco 640x640 (centralizado, aspect ratio mantido)."""
    try:
        img = Image.open(img_path).convert("RGBA")
        # Canvas branco
        canvas = Image.new("RGB", (size, size), (255, 255, 255))
        # Thumbnail preserva aspect ratio
        img.thumbnail((size, size), Image.LANCZOS)
        # Centraliza no canvas
        offset_x = (size - img.width) // 2
        offset_y = (size - img.height) // 2
        # Compor respeitando alpha
        canvas.paste(img, (offset_x, offset_y), img if img.mode == "RGBA" else None)
        canvas.save(output_path, "PNG")
        return True
    except Exception as e:
        print(f"  ⚠️  Imagem inválida: {img_path.name} → {e}")
        return False


def generate_label(class_id: int) -> str:
    """Gera label YOLO: objeto centrado ocupando 100% da imagem."""
    return f"{class_id} 0.500000 0.500000 1.000000 1.000000"


def sanitize_name(name: str) -> str:
    """Remove espaços e caracteres problemáticos do nome."""
    return name.replace(" ", "-").replace("_", "-").lower()


def build_output_name(cloud: str, category: str, original_stem: str) -> str:
    """Monta nome final: {cloud}_{category}_{stem}"""
    cat_clean = sanitize_name(category)
    stem_clean = original_stem.lower().replace(" ", "-")
    return f"{cloud}_{cat_clean}_{stem_clean}"


print("✅ Funções utilitárias definidas.")

✅ Funções utilitárias definidas.


In [6]:
"""
=============================================================
CELL 3: Coletores — Varredura por cloud
=============================================================
Cada coletor retorna lista de tuplas: (file_path, cloud, category, class_id)
"""

def collect_aws() -> list:
    """
    AWS: Architecture-Service-Icons/Arch_*/64/*.svg + Resource-Icons/Res_*/*.svg
    Tratamento especial: Security-Identity-Compliance usa keywords.
    """
    results = []
    
    # --- Architecture Service Icons ---
    arch_dir = AWS_ICONS / "Architecture-Service-Icons"
    if arch_dir.exists():
        for category_folder in sorted(arch_dir.iterdir()):
            if not category_folder.is_dir():
                continue
            # Extrai nome da categoria: "Arch_Compute" → "compute"
            raw_name = category_folder.name
            if raw_name.startswith("Arch_"):
                cat_name = raw_name[5:]  # Remove "Arch_"
            else:
                continue
            
            cat_lower = cat_name.lower()
            
            # Pasta 64/ contém os ícones em melhor resolução
            icons_dir = category_folder / "64"
            if not icons_dir.exists():
                continue
            
            for f in sorted(icons_dir.iterdir()):
                if f.suffix.lower() != ".svg":
                    continue
                
                # Resolve class_id
                class_id = _resolve_aws_class(cat_lower, f.stem)
                if class_id is not None:
                    results.append((f, "aws", cat_name, class_id))
    
    # --- Resource Icons ---
    res_dir = AWS_ICONS / "Resource-Icons"
    if res_dir.exists():
        for category_folder in sorted(res_dir.iterdir()):
            if not category_folder.is_dir():
                continue
            raw_name = category_folder.name
            if raw_name.startswith("Res_"):
                cat_name = raw_name[4:]  # Remove "Res_"
            else:
                continue
            
            cat_lower = cat_name.lower()
            
            for f in sorted(category_folder.iterdir()):
                if f.suffix.lower() != ".svg":
                    continue
                
                class_id = _resolve_aws_class(cat_lower, f.stem)
                if class_id is not None:
                    results.append((f, "aws", cat_name, class_id))
    
    return results


def _resolve_aws_class(cat_lower: str, filename_stem: str) -> int | None:
    """
    Resolve a classe para um arquivo AWS.
    Security-Identity-Compliance usa keyword matching (não mapeamento direto).
    """
    if cat_lower == "security-identity-compliance":
        stem_lower = filename_stem.lower()
        # Gateway keywords: WAF, Shield, KMS, Firewall
        for kw in AWS_SECURITY_KEYWORDS_GATEWAY:
            if kw in stem_lower:
                return 1  # gateway
        # Actor keywords: Cognito, Directory, IAM, Identity
        for kw in AWS_SECURITY_KEYWORDS_ACTOR:
            if kw in stem_lower:
                return 0  # actor
        # Não encaixa → ignora
        return None
    
    # Mapeamento direto para outras pastas
    return FOLDER_TO_CLASS.get(cat_lower, None)


def collect_azure() -> list:
    """Azure: Icons/{category}/*.svg — mapeamento direto."""
    results = []
    if not AZURE_ICONS.exists():
        return results
    
    for category_folder in sorted(AZURE_ICONS.iterdir()):
        if not category_folder.is_dir():
            continue
        cat_name = category_folder.name
        cat_lower = cat_name.lower()
        
        class_id = FOLDER_TO_CLASS.get(cat_lower, None)
        if class_id is None:
            continue
        
        for f in sorted(category_folder.iterdir()):
            if f.suffix.lower() == ".svg":
                results.append((f, "azure", cat_name, class_id))
    
    return results


def collect_gcp() -> list:
    """
    GCP Category: Category/{name}/SVG/*.svg — mapeamento direto
    GCP Unique: Unique/{service}/SVG/*.svg — keyword mapping
    """
    results = []
    
    # --- Category ---
    if GCP_CATEGORY.exists():
        for category_folder in sorted(GCP_CATEGORY.iterdir()):
            if not category_folder.is_dir():
                continue
            cat_name = category_folder.name
            cat_lower = cat_name.lower()
            
            class_id = FOLDER_TO_CLASS.get(cat_lower, None)
            if class_id is None:
                continue
            
            svg_dir = category_folder / "SVG"
            if not svg_dir.exists():
                continue
            
            for f in sorted(svg_dir.iterdir()):
                if f.suffix.lower() == ".svg":
                    results.append((f, "gcp", cat_name, class_id))
    
    # --- Unique ---
    if GCP_UNIQUE.exists():
        for service_folder in sorted(GCP_UNIQUE.iterdir()):
            if not service_folder.is_dir():
                continue
            service_name = service_folder.name
            
            class_id = GCP_UNIQUE_MAPPING.get(service_name, None)
            if class_id is None:
                continue
            
            svg_dir = service_folder / "SVG"
            if not svg_dir.exists():
                continue
            
            for f in sorted(svg_dir.iterdir()):
                if f.suffix.lower() == ".svg":
                    results.append((f, "gcp", service_name, class_id))
    
    return results


# Testa coleta
aws_files = collect_aws()
azure_files = collect_azure()
gcp_files = collect_gcp()

print(f"📦 Arquivos coletados:")
print(f"   AWS:   {len(aws_files)}")
print(f"   Azure: {len(azure_files)}")
print(f"   GCP:   {len(gcp_files)}")
print(f"   TOTAL: {len(aws_files) + len(azure_files) + len(gcp_files)}")

📦 Arquivos coletados:
   AWS:   303
   Azure: 315
   GCP:   21
   TOTAL: 639


In [7]:
"""
=============================================================
CELL 4: Pipeline Principal — Conversão + Split + Salvamento
=============================================================
"""

def process_all():
    """Pipeline completo: coleta → converte → split 80/20 → salva."""
    
    # 1. Coleta todos os arquivos
    all_files = collect_aws() + collect_azure() + collect_gcp()
    print(f"🔍 Total de arquivos para processar: {len(all_files)}")
    
    # 2. Cria diretórios de saída
    for split in ["train", "val"]:
        (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
    
    # 3. Processa cada arquivo → converte para PNG 640x640
    processed = []  # (output_name, class_id, png_bytes_path_temp)
    skipped = 0
    errors = 0
    class_counts = defaultdict(int)
    
    # Pasta temporária para PNGs convertidos
    tmp_dir = OUTPUT_DIR / "_tmp_processing"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    
    for i, (file_path, cloud, category, class_id) in enumerate(all_files):
        output_name = build_output_name(cloud, category, file_path.stem)
        tmp_png = tmp_dir / f"{output_name}.png"
        
        # Converte SVG ou redimensiona PNG/JPG
        if file_path.suffix.lower() == ".svg":
            success = convert_svg_to_png(file_path, tmp_png)
        else:
            success = resize_image_to_canvas(file_path, tmp_png)
        
        if success:
            processed.append((output_name, class_id, tmp_png))
            class_counts[class_id] += 1
        else:
            errors += 1
        
        # Progress log a cada 100 arquivos
        if (i + 1) % 100 == 0:
            print(f"  ... processados {i + 1}/{len(all_files)}")
    
    print(f"\n✅ Conversão concluída:")
    print(f"   Sucesso: {len(processed)}")
    print(f"   Erros:   {errors}")
    print(f"   Distribuição por classe:")
    for cid, count in sorted(class_counts.items()):
        print(f"     [{cid}] {CLASS_NAMES[cid]:12s} → {count} imagens")
    
    # 4. Shuffle e split 80/20
    random.seed(RANDOM_SEED)
    random.shuffle(processed)
    
    split_idx = int(len(processed) * SPLIT_RATIO)
    train_set = processed[:split_idx]
    val_set = processed[split_idx:]
    
    print(f"\n📂 Split:")
    print(f"   Train: {len(train_set)} imagens")
    print(f"   Val:   {len(val_set)} imagens")
    
    # 5. Move arquivos para pastas finais
    for split_name, dataset in [("train", train_set), ("val", val_set)]:
        img_dir = OUTPUT_DIR / split_name / "images"
        lbl_dir = OUTPUT_DIR / split_name / "labels"
        
        for output_name, class_id, tmp_png in dataset:
            # Move PNG
            final_png = img_dir / f"{output_name}.png"
            shutil.move(str(tmp_png), str(final_png))
            
            # Gera label .txt
            label_path = lbl_dir / f"{output_name}.txt"
            label_path.write_text(generate_label(class_id))
    
    # 6. Limpa pasta temporária
    shutil.rmtree(tmp_dir, ignore_errors=True)
    
    # 7. Gera data.yaml para YOLOv8
    yaml_content = f"""train: train/images
val: val/images
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
    (OUTPUT_DIR / "data.yaml").write_text(yaml_content)
    
    print(f"\n🎯 Dataset gerado em: {OUTPUT_DIR}")
    print(f"   data.yaml criado com {len(CLASS_NAMES)} classes.")
    
    return len(processed), errors


# ═══════════════════════════════════════════════════════════
# EXECUTA O PIPELINE
# ═══════════════════════════════════════════════════════════
total, errs = process_all()

🔍 Total de arquivos para processar: 639
  ... processados 100/639
  ... processados 200/639
  ... processados 300/639
  ... processados 400/639
  ... processados 500/639
  ... processados 600/639

✅ Conversão concluída:
   Sucesso: 639
   Erros:   0
   Distribuição por classe:
     [0] actor        → 169 imagens
     [1] gateway      → 130 imagens
     [2] compute      → 110 imagens
     [3] database     → 81 imagens
     [4] storage      → 109 imagens
     [5] message_bus  → 40 imagens

📂 Split:
   Train: 511 imagens
   Val:   128 imagens

🎯 Dataset gerado em: /Users/pedromarquardt/dev/estudos/tech-challenger/archguard-ai/ArchGuard-AI/datasets/archguard
   data.yaml criado com 6 classes.


In [8]:
"""
=============================================================
CELL 5: Verificação — Valida integridade do dataset
=============================================================
"""

def verify_dataset():
    """Verifica se o dataset está íntegro."""
    print("🔎 Verificação do Dataset\n")
    
    for split in ["train", "val"]:
        img_dir = OUTPUT_DIR / split / "images"
        lbl_dir = OUTPUT_DIR / split / "labels"
        
        images = sorted(img_dir.glob("*.png"))
        labels = sorted(lbl_dir.glob("*.txt"))
        
        print(f"  [{split}]")
        print(f"    Imagens: {len(images)}")
        print(f"    Labels:  {len(labels)}")
        
        # Verifica correspondência 1:1
        img_stems = {f.stem for f in images}
        lbl_stems = {f.stem for f in labels}
        
        missing_labels = img_stems - lbl_stems
        missing_images = lbl_stems - img_stems
        
        if missing_labels:
            print(f"    ❌ Imagens sem label: {len(missing_labels)}")
        if missing_images:
            print(f"    ❌ Labels sem imagem: {len(missing_images)}")
        if not missing_labels and not missing_images:
            print(f"    ✅ Correspondência 1:1 OK")
        
        # Verifica tamanho de uma amostra
        if images:
            sample = Image.open(images[0])
            print(f"    📐 Amostra ({images[0].name}): {sample.size}")
        
        # Verifica conteúdo de um label
        if labels:
            content = labels[0].read_text().strip()
            print(f"    📝 Label amostra: '{content}'")
        
        print()
    
    # Distribuição por classe no dataset final
    print("  📊 Distribuição por classe (train + val):")
    class_dist = defaultdict(int)
    for split in ["train", "val"]:
        lbl_dir = OUTPUT_DIR / split / "labels"
        for lbl in lbl_dir.glob("*.txt"):
            cid = int(lbl.read_text().strip().split()[0])
            class_dist[cid] += 1
    
    for cid in sorted(class_dist.keys()):
        print(f"    [{cid}] {CLASS_NAMES[cid]:12s} → {class_dist[cid]}")

verify_dataset()

🔎 Verificação do Dataset

  [train]
    Imagens: 511
    Labels:  511
    ✅ Correspondência 1:1 OK
    📐 Amostra (aws_app-integration_arch_amazon-appflow_64.png): (640, 640)
    📝 Label amostra: '5 0.500000 0.500000 1.000000 1.000000'

  [val]
    Imagens: 128
    Labels:  128
    ✅ Correspondência 1:1 OK
    📐 Amostra (aws_app-integration_arch_amazon-mq_64.png): (640, 640)
    📝 Label amostra: '5 0.500000 0.500000 1.000000 1.000000'

  📊 Distribuição por classe (train + val):
    [0] actor        → 169
    [1] gateway      → 130
    [2] compute      → 110
    [3] database     → 81
    [4] storage      → 109
    [5] message_bus  → 40


## Kaggle — Ingestão de Diagramas Reais

Importa o dataset [carlosrian/software-architecture-dataset](https://www.kaggle.com/datasets/carlosrian/software-architecture-dataset) (Pascal VOC) e converte para YOLO, mapeando ~80 labels para as 6 classes STRIDE.

**Fluxo:** `6b` Download → `6d` Mapeamento → `6c` Diagnóstico → `6e/6f` Ingestão

In [11]:
"""
=============================================================
CELL 6b: Download e Cache — Kaggle Dataset
=============================================================
"""
import xml.etree.ElementTree as ET
import kagglehub

print("📥 Baixando dataset do Kaggle (usa cache se já existir)...")
kaggle_path = Path(kagglehub.dataset_download("carlosrian/software-architecture-dataset"))
print(f"   Path: {kaggle_path}")

def find_dataset_dir(base: Path) -> Path:
    """Encontra a pasta que contém os .xml e .png/.jpg."""
    if list(base.glob("*.xml")):
        return base
    for candidate in [base / "src" / "dataset", base / "dataset", base / "src"]:
        if candidate.exists() and list(candidate.glob("*.xml")):
            return candidate
    for xml_file in base.rglob("*.xml"):
        return xml_file.parent
    raise FileNotFoundError(f"Nenhum XML encontrado em {base}")

kaggle_data_dir = find_dataset_dir(kaggle_path)
xml_files = sorted(kaggle_data_dir.glob("*.xml"))
print(f"   Pasta: {kaggle_data_dir}")
print(f"   XMLs encontrados: {len(xml_files)}")

📥 Baixando dataset do Kaggle (usa cache se já existir)...
   Path: /Users/pedromarquardt/.cache/kagglehub/datasets/carlosrian/software-architecture-dataset/versions/1
   Pasta: /Users/pedromarquardt/.cache/kagglehub/datasets/carlosrian/software-architecture-dataset/versions/1/src/dataset/dataset_augmented
   XMLs encontrados: 8700


In [13]:
"""
=============================================================
CELL 6d: Mapeamento KAGGLE → STRIDE + Parser VOC → YOLO
=============================================================
Ajuste os dicts aqui e re-execute esta célula para corrigir mapeamentos.
Depois re-rode a CELL 6c para validar.
"""

# ── MAPEAMENTO DIRETO: label exato → class_id ──
KAGGLE_TO_STRIDE = {
    # ── COMPUTE (2) ──────────────────────────────────────
    "aws_ec2":                    2, "aws_lambda":                2,
    "aws_elastic_beanstalk":      2, "aws_fargate":               2,
    "aws_ecs":                    2, "aws_eks":                   2,
    "aws_batch":                  2, "aws_lightsail":             2,
    "aws_app_runner":             2, "aws_outposts":              2,
    "aws_auto_scaling":           2, "aws_ec2_auto_scaling":      2,
    "aws_ec2_instances":          2, "aws_ec2_instance":          2,
    "aws_amazon_ec2":             2, "aws_amazon_ec2_auto_scaling": 2,
    "aws_elastic_container_service": 2,
    "aws_elastic_container_service_service": 2,
    "aws_elastic_container_service_container_2": 2,
    "aws_amazon_elastic_container_service": 2,
    "aws_amazon_elastic_kubernetes_service": 2,
    "aws_lambda_lambda_function": 2,
    "azure_vm":                   2, "azure_virtual_machine":     2,
    "azure_functions":            2, "azure_function_apps":       2,
    "azure_app_service":          2,
    "azure_container_instances":  2, "azure_kubernetes":          2,
    "azure_kubernetes_services":  2,
    "azure_aks":                  2, "azure_batch":               2,
    "azure_service_fabric":       2, "azure_spring_apps":         2,
    "azure_vm_scale_sets":        2,
    "gcp_compute_engine":         2, "gcp_cloud_run":             2,
    "gcp_cloud_functions":        2, "gcp_gke":                   2,
    "gcp_google_kubernetes_engine": 2,
    "gcp_app_engine":             2, "gcp_anthos":                2,

    # ── DATABASE (3) ─────────────────────────────────────
    "aws_rds":                    3, "aws_dynamodb":              3,
    "aws_amazon_rds":             3, "aws_amazon_dynamodb":       3,
    "aws_aurora":                 3, "aws_redshift":              3,
    "aws_amazon_redshift":        3,
    "aws_elasticache":            3, "aws_amazon_elasticache":    3,
    "aws_neptune":                3,
    "aws_documentdb":             3, "aws_timestream":            3,
    "aws_keyspaces":              3, "aws_memorydb":              3,
    "aws_qldb":                   3,
    "aws_aurora_amazon_rds_instance": 3,
    "aws_dynamodb_table":         3,
    "azure_sql_database":         3, "azure_cosmos_db":           3,
    "azure_sql":                  3, "azure_sql_server":          3,
    "azure_mysql":                3,
    "azure_postgresql":           3, "azure_mariadb":             3,
    "azure_redis_cache":          3, "azure_managed_database":    3,
    "azure_databricks":           3, "azure_synapse":             3,
    "azure_data_explorer":        3, "azure_data_factories":      3,
    "gcp_cloud_sql":              3, "gcp_cloud_spanner":         3,
    "gcp_bigtable":               3, "gcp_firestore":             3,
    "gcp_memorystore":            3, "gcp_bigquery":              3,
    "gcp_alloydb":                3, "gcp_datastore":             3,

    # ── STORAGE (4) ──────────────────────────────────────
    "aws_s3":                     4, "aws_ebs":                   4,
    "aws_efs":                    4, "aws_fsx":                   4,
    "aws_glacier":                4, "aws_storage_gateway":       4,
    "aws_backup":                 4,
    "aws_amazon_simple_storage_service": 4,
    "aws_simple_storage_service_s3_standard": 4,
    "aws_simple_storage_service_bucket": 4,
    "aws_simple_storage_service_object": 4,
    "aws_amazon_elastic_block_store": 4,
    "aws_elastic_block_store":    4,
    "azure_blob_storage":         4, "azure_storage":             4,
    "azure_storage_accounts":     4,
    "azure_disk_storage":         4, "azure_data_lake":           4,
    "azure_file_storage":         4, "azure_netapp":              4,
    "azure_storage_account":      4,
    "gcp_cloud_storage":          4, "gcp_persistent_disk":       4,
    "gcp_filestore":              4,

    # ── GATEWAY (1) ──────────────────────────────────────
    "aws_api_gateway":            1, "aws_cloudfront":            1,
    "aws_amazon_api_gateway":     1, "aws_amazon_cloudfront":     1,
    "aws_elb":                    1, "aws_alb":                   1,
    "aws_nlb":                    1, "aws_route53":               1,
    "aws_amazon_route_53":        1, "aws_route_53_hosted_zone":  1,
    "aws_vpc":                    1, "aws_direct_connect":        1,
    "aws_amazon_virtual_private_cloud": 1,
    "aws_vpc_virtual_private_cloud_vpc": 1,
    "aws_global_accelerator":     1, "aws_transit_gateway":       1,
    "aws_waf":                    1, "aws_shield":                1,
    "aws_network_firewall":       1, "aws_privatelink":           1,
    "aws_elastic_load_balancing": 1,
    "aws_elastic_load_balancing_application_load_balancer": 1,
    "azure_load_balancer":        1, "azure_load_balancers":      1,
    "azure_application_gateway":  1,
    "azure_front_door":           1, "azure_traffic_manager":     1,
    "azure_cdn":                  1, "azure_firewall":            1,
    "azure_dns":                  1, "azure_vnet":                1,
    "azure_virtual_networks":     1,
    "azure_vpn_gateway":          1, "azure_expressroute":        1,
    "azure_bastion":              1, "azure_waf":                 1,
    "azure_network_security_groups": 1,
    "azure_api_management_services": 1,
    "gcp_cloud_load_balancing":   1, "gcp_cloud_cdn":             1,
    "gcp_cloud_dns":              1, "gcp_cloud_armor":           1,
    "gcp_cloud_nat":              1, "gcp_vpc":                   1,
    "gcp_virtual_private_cloud":  1,
    "gcp_apigee":                 1, "gcp_cloud_endpoints":       1,

    # ── MESSAGE BUS (5) ──────────────────────────────────
    "aws_sqs":                    5, "aws_sns":                   5,
    "aws_simple_queue_service_queue": 5,
    "aws_simple_notification_service_topic": 5,
    "aws_eventbridge":            5, "aws_kinesis":               5,
    "aws_mq":                     5, "aws_step_functions":        5,
    "aws_appflow":                5, "aws_msk":                   5,
    "azure_service_bus":          5, "azure_event_hub":           5,
    "azure_event_hubs":           5,
    "azure_event_grid":           5, "azure_logic_apps":          5,
    "azure_queue_storage":        5, "azure_relay":               5,
    "gcp_pubsub":                 5, "gcp_cloud_tasks":           5,
    "gcp_dataflow":               5, "gcp_workflows":             5,
    "gcp_eventarc":               5,

    # ── ACTOR (0) ────────────────────────────────────────
    "aws_iam":                    0, "aws_cognito":               0,
    "aws_identity_and_access_management": 0,
    "aws_identity_access_management_role": 0,
    "aws_key_management_service": 0,
    "aws_directory_service":      0, "aws_sso":                   0,
    "azure_active_directory":     0, "azure_ad_b2c":              0,
    "azure_identity":             0, "azure_entra":               0,
    "azure_key_vaults":           0,
    "gcp_iam":                    0, "gcp_identity_platform":     0,
    "gcp_identity_and_access_management": 0,
    "user":                       0, "client":                    0,
    "mobile":                     0, "actor":                     0,
    "browser":                    0, "device":                    0,
}

# ── FALLBACK POR KEYWORD (ordem importa — mais específico primeiro) ──
KEYWORD_FALLBACK = [
    # STORAGE primeiro (evita "elastic_block" cair em compute por "elastic")
    (["simple_storage_service", "elastic_block_store",
      "storage_account", "storage_gateway", "blob_storage",
      "data_lake", "s3_standard", "s3_bucket", "s3_object",
      "glacier", "fsx", "efs", "ebs", "netapp", "filestore",
      "persistent_disk", "backup"], 4),
    # DATABASE
    (["database", "sql_database", "sql_server", "cosmos_db",
      "dynamodb", "aurora", "redshift", "elasticache", "neptune",
      "documentdb", "timestream", "spanner", "bigtable",
      "firestore", "bigquery", "alloydb", "datastore",
      "memorystore", "memorydb", "redis_cache", "data_factories",
      "rds_instance"], 3),
    # GATEWAY (networking/security boundary)
    (["api_gateway", "load_balancer", "cloudfront", "route_53",
      "route53", "cloud_load_balancing", "front_door",
      "traffic_manager", "cdn", "application_gateway",
      "network_security_group", "virtual_network", "vnet",
      "virtual_private_cloud", "vpc", "vpn_gateway",
      "expressroute", "direct_connect", "bastion",
      "cloud_armor", "cloud_nat", "cloud_dns",
      "api_management", "apigee"], 1),
    # COMPUTE
    (["compute_engine", "cloud_run", "cloud_function", "app_engine",
      "virtual_machine", "vm_scale_set", "kubernetes",
      "container_service", "container_instance",
      "elastic_beanstalk", "fargate", "lightsail",
      "app_runner", "lambda_function", "auto_scaling",
      "ec2_instance", "ec2_auto"], 2),
    # MESSAGE BUS
    (["queue_service", "notification_service", "event_hub",
      "event_grid", "service_bus", "logic_app",
      "step_function", "eventbridge", "kinesis",
      "pubsub", "cloud_tasks", "dataflow", "workflows",
      "eventarc", "relay"], 5),
    # ACTOR (último — keywords genéricas como "identity")
    (["identity_and_access", "access_management",
      "key_management", "key_vault", "cognito",
      "directory_service", "sso", "active_directory",
      "entra", "identity_platform"], 0),
]


def resolve_kaggle_label(label: str) -> int | None:
    """Mapeia um label do Kaggle para class_id STRIDE."""
    normalized = label.strip().lower().replace("-", "_").replace(" ", "_")

    # 1. Busca direta no dicionário
    if normalized in KAGGLE_TO_STRIDE:
        return KAGGLE_TO_STRIDE[normalized]

    # 2. Fallback por keyword (mais específico primeiro)
    for keywords, class_id in KEYWORD_FALLBACK:
        for kw in keywords:
            if kw in normalized:
                return class_id

    # 3. Não mapeável → ignora
    return None


def parse_voc_xml(xml_path: Path) -> list | None:
    """Lê XML Pascal VOC → lista de (class_id, x_center, y_center, w, h) YOLO."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"  ⚠️  XML corrompido: {xml_path.name} → {e}")
        return None

    size_el = root.find("size")
    if size_el is None:
        return None
    img_w = int(size_el.findtext("width", "0"))
    img_h = int(size_el.findtext("height", "0"))
    if img_w == 0 or img_h == 0:
        return None

    annotations = []
    for obj in root.findall("object"):
        label = obj.findtext("name", "").strip()
        class_id = resolve_kaggle_label(label)
        if class_id is None:
            continue

        bndbox = obj.find("bndbox")
        if bndbox is None:
            continue

        xmin = max(0, min(float(bndbox.findtext("xmin", "0")), img_w))
        ymin = max(0, min(float(bndbox.findtext("ymin", "0")), img_h))
        xmax = max(0, min(float(bndbox.findtext("xmax", "0")), img_w))
        ymax = max(0, min(float(bndbox.findtext("ymax", "0")), img_h))

        x_center = ((xmin + xmax) / 2.0) / img_w
        y_center = ((ymin + ymax) / 2.0) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        if w > 0.001 and h > 0.001:
            annotations.append((class_id, x_center, y_center, w, h))

    return annotations if annotations else None


print("✅ Dicionários e funções carregados.")
print(f"   Labels diretos: {len(KAGGLE_TO_STRIDE)}")
print(f"   Grupos de fallback: {len(KEYWORD_FALLBACK)}")

✅ Dicionários e funções carregados.
   Labels diretos: 194
   Grupos de fallback: 6


In [14]:
"""
=============================================================
CELL 6c: Diagnóstico — Scan de todos os labels do Kaggle
=============================================================
Varre todos os XMLs e mostra quais labels existem,
quantos mapeiam para STRIDE e quantos ficam de fora.
RODE ESTA CÉLULA ANTES de processar para validar o mapeamento.
"""
from collections import Counter

# Conta todas as ocorrências de cada label nos XMLs
all_labels = Counter()
for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        for obj in tree.getroot().findall("object"):
            lbl = obj.findtext("name", "").strip()
            if lbl:
                all_labels[lbl] += 1
    except ET.ParseError:
        continue

print(f"📊 Total de labels únicos encontrados: {len(all_labels)}")
print(f"   Total de ocorrências (bboxes): {sum(all_labels.values())}\n")

# Classifica cada label
mapped = {}    # label → (class_id, class_name, count)
unmapped = {}  # label → count

for lbl, count in all_labels.most_common():
    normalized = lbl.strip().lower().replace("-", "_").replace(" ", "_")

    # Busca direta
    if normalized in KAGGLE_TO_STRIDE:
        cid = KAGGLE_TO_STRIDE[normalized]
        mapped[lbl] = (cid, CLASS_NAMES[cid], count, "direto")
        continue

    # Fallback por keyword
    found = False
    for keywords, class_id in KEYWORD_FALLBACK:
        for kw in keywords:
            if kw in normalized:
                mapped[lbl] = (class_id, CLASS_NAMES[class_id], count, f"keyword: '{kw}'")
                found = True
                break
        if found:
            break

    if not found:
        unmapped[lbl] = count

# ── Log mapeados ──
print(f"✅ MAPEADOS: {len(mapped)} labels ({sum(v[2] for v in mapped.values())} bboxes)")
print(f"{'─'*90}")
print(f"  {'Label Kaggle':<45s} │ {'STRIDE':<15s} │ {'Qtd':>5s} │ {'Via'}")
print(f"  {'─'*45}─┼─{'─'*15}─┼─{'─'*5}─┼─{'─'*20}")
for lbl in sorted(mapped.keys()):
    cid, cname, count, via = mapped[lbl]
    marker = "⚠️" if via != "direto" else "  "
    print(f"{marker}{lbl:<45s} │ {cname} ({cid}){'':<5s} │ {count:>5d} │ {via}")

# ── Log NÃO mapeados ──
print(f"\n❌ NÃO MAPEADOS: {len(unmapped)} labels ({sum(unmapped.values())} bboxes)")
print(f"{'─'*60}")
for lbl, count in sorted(unmapped.items(), key=lambda x: -x[1]):
    print(f"   '{lbl}' → {count} ocorrências")

# ── Resumo por classe STRIDE ──
print(f"\n📈 Distribuição prevista por classe STRIDE:")
class_preview = Counter()
for cid, cname, count, via in mapped.values():
    class_preview[cid] += count
for cid in range(len(CLASS_NAMES)):
    print(f"   [{cid}] {CLASS_NAMES[cid]:12s} → {class_preview.get(cid, 0):>6d} bboxes")

📊 Total de labels únicos encontrados: 111
   Total de ocorrências (bboxes): 105666

✅ MAPEADOS: 80 labels (85328 bboxes)
──────────────────────────────────────────────────────────────────────────────────────────
  Label Kaggle                                  │ STRIDE          │   Qtd │ Via
  ──────────────────────────────────────────────┼─────────────────┼───────┼─────────────────────
  aws_amazon_api_gateway                        │ gateway (1)      │  1070 │ direto
  aws_amazon_cloudfront                         │ gateway (1)      │  1260 │ direto
  aws_amazon_dynamodb                           │ database (3)      │  1140 │ direto
  aws_amazon_ec2                                │ compute (2)      │   990 │ direto
  aws_amazon_ec2_auto_scaling                   │ compute (2)      │  1180 │ direto
  aws_amazon_elastic_block_store                │ storage (4)      │  1270 │ direto
  aws_amazon_elastic_container_service          │ compute (2)      │  1110 │ direto
  aws_amazon_elastic_k

In [15]:
print(f"actor bboxes: {class_preview.get(0, 0)}")

actor bboxes: 6358


In [16]:
"""
=============================================================
CELL 6e: Pipeline — Ingestão Mista (Append)
=============================================================
"""

def ingest_kaggle():
    """Processa o dataset Kaggle e faz append nas pastas do ArchGuard."""
    pairs = []
    unmapped_labels = defaultdict(int)

    for xml_file in xml_files:
        img_file = None
        for ext in [".png", ".jpg", ".jpeg"]:
            candidate = xml_file.with_suffix(ext)
            if candidate.exists():
                img_file = candidate
                break
        if img_file is None:
            continue

        annotations = parse_voc_xml(xml_file)
        if annotations is None:
            try:
                tree = ET.parse(xml_file)
                for obj in tree.getroot().findall("object"):
                    lbl = obj.findtext("name", "").strip()
                    if resolve_kaggle_label(lbl) is None:
                        unmapped_labels[lbl] += 1
            except Exception:
                pass
            continue

        pairs.append((img_file, annotations))

    print(f"📦 Kaggle: {len(pairs)} pares válidos (imagem + anotações YOLO)")

    if unmapped_labels:
        print(f"\n  ⚠️  Labels não mapeados (ignorados):")
        for lbl, count in sorted(unmapped_labels.items(), key=lambda x: -x[1])[:20]:
            print(f"      '{lbl}' → {count} ocorrências")

    # Shuffle e split 80/20
    random.seed(RANDOM_SEED)
    random.shuffle(pairs)
    split_idx = int(len(pairs) * SPLIT_RATIO)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]

    added = {"train": 0, "val": 0}
    class_counts = defaultdict(int)

    for split_name, split_pairs in [("train", train_pairs), ("val", val_pairs)]:
        img_dir = OUTPUT_DIR / split_name / "images"
        lbl_dir = OUTPUT_DIR / split_name / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_file, annotations in split_pairs:
            safe_name = f"kaggle_aug_{img_file.stem}"
            dest_img = img_dir / f"{safe_name}{img_file.suffix}"
            dest_lbl = lbl_dir / f"{safe_name}.txt"

            if not dest_img.exists():
                shutil.copy2(str(img_file), str(dest_img))

            label_lines = []
            for class_id, xc, yc, w, h in annotations:
                label_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
                class_counts[class_id] += 1
            dest_lbl.write_text("\n".join(label_lines))
            added[split_name] += 1

    print(f"\n✅ Ingestão Kaggle concluída:")
    print(f"   Train: +{added['train']} imagens")
    print(f"   Val:   +{added['val']} imagens")

    print(f"\n  📊 Distribuição das anotações Kaggle por classe:")
    for cid in sorted(class_counts.keys()):
        print(f"    [{cid}] {CLASS_NAMES[cid]:12s} → {class_counts[cid]} bboxes")

    print(f"\n📂 Dataset Consolidado (ícones + Kaggle):")
    for split in ["train", "val"]:
        img_count = len(list((OUTPUT_DIR / split / "images").glob("*")))
        lbl_count = len(list((OUTPUT_DIR / split / "labels").glob("*.txt")))
        print(f"   [{split}] {img_count} imagens / {lbl_count} labels")

print("✅ Função ingest_kaggle() definida. Execute a próxima célula para rodar.")

✅ Função ingest_kaggle() definida. Execute a próxima célula para rodar.


In [17]:
# ═══════════════════════════════════════════════════════════
# EXECUTA INGESTÃO KAGGLE
# ═══════════════════════════════════════════════════════════
ingest_kaggle()

📦 Kaggle: 8700 pares válidos (imagem + anotações YOLO)

✅ Ingestão Kaggle concluída:
   Train: +6960 imagens
   Val:   +1740 imagens

  📊 Distribuição das anotações Kaggle por classe:
    [0] actor        → 6358 bboxes
    [1] gateway      → 18810 bboxes
    [2] compute      → 23590 bboxes
    [3] database     → 17050 bboxes
    [4] storage      → 10810 bboxes
    [5] message_bus  → 8710 bboxes

📂 Dataset Consolidado (ícones + Kaggle):
   [train] 7471 imagens / 7471 labels
   [val] 1868 imagens / 1868 labels
